<a href="https://colab.research.google.com/github/rburchf1/AI102Challenges/blob/main/assignment1_sentiment_ryan_burchfield.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 — Sentiment Analysis: Classical ML vs. DL vs. LLM

**Ryan Burchfield**

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Overview & dataset selection

Twitter US Airline Sentiment

I selected the Twitter dataset because the sentiment analysis contains 3 options: positive, negative, and neutral. Determining neutrality seems more challenging, in terms of semantic analysis, than a binary setup. In addition, the tertiary framework is more comparable to the type of analysis I might conduct at work.

## 2. Load & inspect data
Use Hugging Face `datasets` or local CSV to load your dataset. Show class distribution and a few samples.

In [3]:
# TODO: Install/Import basics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# from utils.data_utils import basic_clean # Commented out, as 'utils' module is not found
print('OK: libraries imported')

OK: libraries imported


In [17]:
# Load Twitter Dataset
file_path = '/content/drive/MyDrive/Tweets.csv'
try:
  ds = pd.read_csv(file_path)
  print('File loaded successfully.')
except FileNotFoundError:
  print(f'File not found at path: {file_path}. Please check the file path.')
except Exception as e:
  print(f'An error occurred while loading the file: {e}')

#Class distribution and a few examples
print("DataSet Info:")
ds.info()
print("\nDataSet Description:")
display(ds.describe(include='all'))
print("\nFirst 5 rows of the DataSet:")
display(ds.head())

File loaded successfully.
DataSet Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  object 
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object 
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object 
 6   airline_sentiment_gold        40 non-null     object 
 7   name                          14640 non-null  object 
 8   negativereason_gold           32 non-null     object 
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  object 
 11  tweet_coord                   1019 non-null   object 
 12  tweet_created       

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
count,1.464000e+04,14640,14640.000000,9178,10522.000000,14640,40,14640,32,14640.000000,14640,1019,14640,9907,9820
unique,NaN,3,NaN,10,NaN,6,3,7701,13,NaN,14427,832,14247,3081,85
top,NaN,negative,NaN,Customer Service Issue,NaN,United,negative,JetBlueNews,Customer Service Issue,NaN,@united thanks,"[0.0, 0.0]",2015-02-24 09:54:34 -0800,"Boston, MA",Eastern Time (US & Canada)
freq,NaN,9178,NaN,2910,NaN,3822,32,63,12,NaN,6,164,5,157,3744
mean,5.692184e+17,NaN,0.900169,NaN,0.638298,NaN,NaN,NaN,NaN,0.082650,NaN,NaN,NaN,NaN,NaN
std,7.791112e+14,NaN,0.162830,NaN,0.330440,NaN,NaN,NaN,NaN,0.745778,NaN,NaN,NaN,NaN,NaN
min,5.675883e+17,NaN,0.335000,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
25%,5.685592e+17,NaN,0.692300,NaN,0.360600,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
50%,5.694779e+17,NaN,1.000000,NaN,0.670600,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
75%,5.698905e+17,NaN,1.000000,NaN,1.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN



First 5 rows of the DataSet:


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [22]:
# Evaluate the dataset for balance
print('Sentiment class distribution:')
display(y.value_counts())

Sentiment class distribution:


,count
airline_sentiment,
negative,9178
neutral,3099
positive,2363


COMMENTS ON DATASET BALANCE

As one might expect, the dataset is imbalanced towared the negative. It seems that customers are most likely to spend effort to tweet an experience when it is negative and accept a neutral or good experience as a fair trade since the paid for a service.

## 3. Preprocessing
Start simple: lowercase, remove URLs/usernames (if any), strip spaces, and tokenize if needed. Consider whether to keep emojis.

In [25]:
#EMOJIS
#Detect if there are emojies in the dataset

import re

# Regex to detect most common emojis
# This regex covers a broad range of Unicode emoji blocks and sequences.
# It might not catch all edge cases or very new emojis, but it's a good starting point.
emoji_pattern = re.compile(
    "["  # Start character set
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F300-\U0001F5FF"  # Miscellaneous Symbols and Pictographs
    "\U0001F680-\U0001F6FF"  # Transport and Map Symbols
    "\U0001F1E0-\U0001F1FF"  # Regional indicator symbols
    "\U00002600-\U000026FF"  # Miscellaneous Symbols
    "\U00002700-\U000027BF"  # Dingbats
    "]+"
)

def contains_emoji(text):
    return bool(emoji_pattern.search(str(text)))

# Sample some texts to check for emojis
print("Checking for emojis in sample texts:")
found_emoji = False
for i, text in enumerate(x.sample(n=10, random_state=42)):
    if contains_emoji(text):
        print(f"  Text {i+1} (contains emoji): {text}")
        found_emoji = True
    else:
        print(f"  Text {i+1} (no emoji): {text}")

if not found_emoji:
    print("  No emojis found in the selected sample of texts.")
else:
    print("  Emojis were found in the sample texts.")

print("\nNow checking the full dataset (this might take a moment if the dataset is large):")
# Check the entire 'text' column for emojis
num_texts_with_emojis = x.apply(contains_emoji).sum()

if num_texts_with_emojis > 0:
    print(f"The 'text' column contains emojis. Found {num_texts_with_emojis} texts with emojis.")
    print("It would be beneficial to decide whether to remove or preserve them during preprocessing based on the task.")
else:
    print("The 'text' column does not appear to contain emojis.")

Checking for emojis in sample texts:
  Text 1 (no emoji): @SouthwestAir you're my early frontrunner for best airline! #oscars2016
  Text 2 (no emoji): @USAirways how is it that my flt to EWR was Cancelled Flightled yet flts to NYC from USAirways are still flying?
  Text 3 (no emoji): @JetBlue what is going on with your BDL to DCA flights yesterday and today?! Why is every single one getting delayed?
  Text 4 (no emoji): @JetBlue do they have to depart from Washington, D.C.??
  Text 5 (no emoji): @JetBlue I can probably find some of them. Are the ticket #s on there?
  Text 6 (no emoji): @united still waiting to hear back. My wallet was stolen from one of your planes so would appreciate a resolution here
  Text 7 (no emoji): @united Yes my flight was rebooked. I'm just losing trust in you if I want to get anywhere on time.
  Text 8 (no emoji): @JetBlue Thank you ! What about Paris ? Could we arrange something from there ?
  Text 9 (no emoji): @united not 100% sure, however my ticket incl

EMOJIS ANALYSIS AND DECISION ON WHETHER TO KEEP OR REMOVE EMOJIS WHEN CLEANING DATA

The dataset contains 487 entries with emojis, which represent only ~3% of the tweets. Therefore, emojis will be removed.

In [24]:
# Split data for training and validation and testing
from sklearn.model_selection import train_test_split

#FIRST SPLIT
#80% training + validation, 20% for test

x = ds['text']
y = ds['airline_sentiment']

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#SECOND SPLIT
#Take 12.5% of the 80% for validation
#12.5% of 80% = 10% of the original dataset

x_train, x_val, y_train, y_val = train_test_split(
    x, y,
    test_size=0.125,
    random_state=42,
    stratify=y
)

In [27]:
#Define and apply a clean function

def basic_clean(text):
    text = str(text).lower()  # Convert to string and lowercase

    # Remove URLs (http/https links)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove mentions (@usernames)
    text = re.sub(r'@\w+', '', text)

    # Remove emojis using the previously defined pattern
    text = emoji_pattern.sub(r'', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the cleaning function
clean_x_train = x_train.apply(basic_clean)
clean_x_val = x_val.apply(basic_clean)
clean_x_test = x_test.apply(basic_clean)

print("Original text sample:", x_train.iloc[0])
print("Cleaned text sample:", clean_x_train.iloc[0])

print("\nFirst 5 cleaned training texts:")
display(clean_x_train.head())

Original text sample: @VirginAmerica Can't bring up my reservation online using Flight Booking Problems code
Cleaned text sample: can't bring up my reservation online using flight booking problems code

First 5 cleaned training texts:


,text
86,can't bring up my reservation online using fli...
14047,educate bohol is a 501(c)(3) w/all volunteer s...
3642,i mean is there a real live person somewhere i...
2356,how about plowing the snow at a gate before th...
5455,i met my twitter friend waiting outside the tr...


## 4. Model A: Classical ML (TF–IDF + Logistic Regression or SVM)

In [ ]:
# TODO: Classical baseline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# X_train, X_val, y_train, y_val = train_test_split(clean_train, train_labels, test_size=0.2, random_state=42, stratify=train_labels)
# tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1,2))
# Xtr = tfidf.fit_transform(X_train)
# Xva = tfidf.transform(X_val)
# clf = LogisticRegression(max_iter=200)
# clf.fit(Xtr, y_train)
# preds = clf.predict(Xva)
# print(classification_report(y_val, preds, digits=4))


## 5. Model B: Deep Learning (LSTM or 1D-CNN)

In [ ]:
# TODO: Build a small LSTM or 1D-CNN (PyTorch)
# Outline: build vocab -> encode/pad -> DataLoader -> model/train/eval
import torch, torch.nn as nn, torch.optim as optim
# Define your model class and training loop here


## 6. Model C: LLM (BERT/DistilBERT fine-tuning)

In [ ]:
# TODO: Use Hugging Face Transformers Trainer API for a quick run
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Example skeleton (you must adapt variables):
# tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
# train_ds = Dataset.from_dict({'text': clean_train, 'label': train_labels})
# test_ds  = Dataset.from_dict({'text': clean_test,  'label': test_labels})
# def tokenize(b):
#     return tok(b['text'], truncation=True, padding=True, max_length=256)
# ds_tok = DatasetDict({'train': train_ds.map(tokenize, batched=True), 'test': test_ds.map(tokenize, batched=True)})
# model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     preds = np.argmax(logits, axis=1)
#     p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
#     acc = accuracy_score(labels, preds)
#     return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}
# args = TrainingArguments(output_dir='./out', num_train_epochs=2, per_device_train_batch_size=16, per_device_eval_batch_size=32, evaluation_strategy='epoch', learning_rate=2e-5, weight_decay=0.01)
# trainer = Trainer(model=model, args=args, train_dataset=ds_tok['train'], eval_dataset=ds_tok['test'], compute_metrics=compute_metrics)
# trainer.train(); trainer.evaluate()


## 7. Results comparison
Create a table of metrics for all models (Accuracy/Precision/Recall/F1).

## 8. Error analysis & reflection
Show misclassified examples and discuss patterns. Connect to challenges (context, emojis, domain shift, explainability).

## 9. Reproducibility notes (how to run)
List environment, hardware (CPU/GPU), and exact commands used.